# 1. Production-Level Order Fulfillment Engine

---

## Task

Design and implement a production-ready Python function that processes an e-commerce order and returns a validated fulfillment summary.

This is not a basic calculation problem. It involves:

* Strict validation
* Business rule handling
* Clean design
* Extensibility

---

## Objective

Build a single reusable function that:

* Accepts order and customer details
* Processes multiple products (price + quantity)
* Applies tax, shipping, and coupon logic
* Handles invalid inputs and edge cases
* Returns a structured fulfillment summary

---

## Function Signature (Expected)

from typing import Dict, Any, Tuple
```python
def process_order(
    order_id: str,
    customer_name: str,
    *items: Tuple[str, float, int],
    **kwargs: Any
) -> Dict[str, Any]:
    ...
```

---

## Input

### Required Arguments

* order_id (str) – Unique order identifier
* customer_name (str) – Customer name

---

### Variable Positional Arguments (*items)

Each item must be a tuple:
(product_name: str, price: float, quantity: int)

Example:
("Laptop", 50000, 1), ("Mouse", 500, 2)

---

### Keyword Arguments (**kwargs)

Optional configuration:

* tax_rate (float, default: 18)
* shipping_fee (float, default: 100)
* coupon_code (str, optional)
* round_off (bool, default: True)

Additional metadata:

* country
* order_date
* etc.

---

## Coupon Logic

| Coupon Code | Rule                             |
| ----------- | -------------------------------- |
| SAVE10      | 10% discount on subtotal         |
| FLAT500     | ₹500 discount if subtotal ≥ 5000 |
| FREESHIP    | Shipping fee becomes 0           |

### Important Rules:

* Only one coupon allowed
* Invalid coupon → raise error
* Coupon logic must be applied correctly and safely

---

## Output

The function must return a dictionary:

{
"order_id": "ORD-101",
"customer": "John Doe",
"items_total": 51000.0,
"tax_amount": 9180.0,
"discount_amount": 500.0,
"shipping_fee": 0.0,
"final_amount": 59680.0,
"applied_coupon": "FLAT500",
"metadata": {
"country": "India",
"order_date": "2026-03-17"
}
}

---

## Functional Requirements

### Validation

* Ensure order_id and customer_name are non-empty strings
* Ensure at least one item is provided
* Validate each item:

  * Must be a tuple of length 3
  * product_name must be string
  * price must be int/float ≥ 0
  * quantity must be int > 0

### Configuration Validation

* tax_rate must be between 0 and 100
* shipping_fee must be ≥ 0
* coupon_code must be valid (if provided)

### Business Logic

* items_total = sum(price × quantity)
* tax_amount = items_total × tax_rate / 100

Apply coupon:

* SAVE10 → percentage discount
* FLAT500 → conditional flat discount
* FREESHIP → shipping becomes 0

Final calculation:
final_amount = items_total + tax_amount + shipping_fee - discount_amount

### Output Rules

* Return structured dictionary (no print statements)
* Round values to 2 decimal places if round_off=True
* Keep metadata separate from config values

---

## Constraints

* No global variables
* No mutation of input arguments
* No hardcoding outside function logic
* Must use *items and **kwargs
* Must raise meaningful exceptions
* Must follow clean naming conventions
* Must include proper docstring and typing

---

## Edge Cases You Must Handle

* Empty items → error
* Invalid tuple structure → error
* Negative price → error
* Zero/negative quantity → error
* Invalid coupon → error
* Coupon applied but conditions not met → handle correctly
* Missing optional fields → fallback to defaults

---

## Evaluation Criteria

You will be judged on:

* Code readability
* Validation depth
* Error handling quality
* Separation of concerns
* Extensibility (especially coupon logic)
* Clean structure under complexity

---

## Warning

If your code:

* Works only for happy paths
* Mixes validation and business logic messily
* Hardcodes coupon logic poorly

Then it’s not production-level — it’s beginner-level.

---

## Goal

Write code that survives:

* Messy real-world inputs
* Changing requirements
* Additional coupons without breaking logic

If your code collapses when complexity increases, it's not solid.

In [2]:
from typing import Dict, List, Tuple, Any

In [9]:
def process_order(order_id: str, customer_name: str, *items: Tuple[str, float, int], **kwargs: Any) -> Dict[str, Any]:
    """
    Process an e-commerce order and return a validated fulfillment summary.

    Args:
        order_id: Unique order identifier
        customer_name: Name of the customer
        *items: Variable number of items as tuples (product_name, price, quantity)
        **kwargs: Optional configuration and metadata

    Returns:
        Dictionary containing fulfillment summary

    Raises:
        ValueError: For invalid inputs or business rule violations
        TypeError: For incorrect types
    """

    config_keys = {"tax_rate", "shipping_fee", "coupon_code", "round_off"}
    tax_rate = kwargs.get("tax_rate", 18)
    shipping_fee = kwargs.get("shipping_fee", 100)
    coupon_code = kwargs.get("coupon_code", None)
    round_off = kwargs.get("round_off", True)

    metadata = {k: v for k, v in kwargs.items() if k not in config_keys}

    if not isinstance(order_id,str):
        raise ValueError("Invalid Order id")
    if not isinstance(customer_name, str):
        raise ValueError("Invalid Customer Name")

    if not items:
        raise ValueError("No items provided")
    
    normalized_items = []
    
    for idx, item in enumerate(items):
        if not isinstance(item, tuple) or len(item) != 3:
            raise ValueError(f"Invalid item at index {idx}")
    
        name, price, quantity = item
    
        if not name or not isinstance(name, str):
            raise ValueError(f"Invalid name at index {idx}")
    
        if price < 0:
            raise ValueError(f"Price must be >= 0 for {name}")
    
        if quantity <= 0:
            raise ValueError(f"Quantity must be > 0 for {name}")
    
        normalized_items.append({
            "name": name.strip(),
            "price": float(price),
            "quantity": quantity
        })

    if not (0 <= tax_rate <= 100):
        raise ValueError("tax_rate must be between 0 and 100")
    
    if shipping_fee < 0:
        raise ValueError("shipping_fee cannot be negative")
    
    valid_coupons = {"SAVE10", "FLAT500", "FREESHIP"}
    
    if coupon_code and coupon_code not in valid_coupons:
        raise ValueError("Invalid coupon_code")

    def apply_save10(subtotal, shipping):
        return subtotal * 0.10, shipping
        
    def apply_flat500(subtotal,shipping):
        if subtotal >= 5000:
            return 500.0, shipping
        else:
            return 0.0, shipping
            
    def apply_freeship(subtotal: float, shipping: float):
        return 0.0, 0.0
        
    coupon_handlers = {
        "SAVE10": apply_save10,
        "FLAT500": apply_flat500,
        "FREESHIP": apply_freeship
    }
    items_total = sum(item["price"] * item["quantity"] for item in normalized_items)
    tax_amount = (items_total * tax_rate) / 100
    discount_amount = 0.0
    applied_coupon = None
    if coupon_code:
        handler = coupon_handlers[coupon_code]
        discount_amount, shipping_fee = handler(items_total, shipping_fee)
        applied_coupon = coupon_code
    final_amount = items_total + tax_amount + shipping_fee - discount_amount
    if round_off:
        items_total = round(items_total, 2)
        tax_amount = round(tax_amount, 2)
        discount_amount = round(discount_amount, 2)
        shipping_fee = round(shipping_fee, 2)
        final_amount = round(final_amount, 2)
    return {
        "order_id": order_id,
        "customer": customer_name,
        "items_total": items_total,
        "tax_amount": tax_amount,
        "discount_amount": discount_amount,
        "shipping_fee": shipping_fee,
        "final_amount": final_amount,
        "applied_coupon": applied_coupon,
        "metadata": metadata
    }

In [11]:
process_order(
    "ORD-101",
    "Sarthak",
    ("Laptop", 50000, 1),
    ("Mouse", 500, 2),
    tax_rate=10,
    shipping_fee=200,
    coupon_code="FLAT500",
    country="India",
    order_date="2026-03-17"
)

{'order_id': 'ORD-101',
 'customer': 'Sarthak',
 'items_total': 51000.0,
 'tax_amount': 5100.0,
 'discount_amount': 500.0,
 'shipping_fee': 200,
 'final_amount': 55800.0,
 'applied_coupon': 'FLAT500',
 'metadata': {'country': 'India', 'order_date': '2026-03-17'}}